# gcn vs gat: does graph noise break them differently?
bigger pilot experiment on pubmed (19,717 nodes) - compare gcn and gat when we add or remove random edges, look at the graph itself, and compare the two models properly

install pytorch geometric

In [1]:
!pip install torch_geometric -q


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


import everything we need

In [2]:
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import confusion_matrix
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATConv
from torch_geometric.utils import to_networkx

/Users/ashishmeshram/Desktop/Master-Project/My-Implementation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


small helper: build a legend that maps each class number to its color, so every graph picture is readable

In [3]:
def class_legend_handles(num_classes):
    cmap = plt.cm.tab10
    return [mpatches.Patch(color=cmap(c), label=f"class {c}") for c in range(num_classes)]

set seed and pick device

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cpu


load the pubmed dataset - bigger than cora, so this counts as a proper large-scale test

In [5]:
dataset = Planetoid(root="data/Planetoid", name="PubMed")
data_cpu = dataset[0]
data = data_cpu.clone().to(device)
num_classes = dataset.num_classes

Processing...
Done!


look at basic info about the graph

In [6]:
print(data)
print("num nodes:", data.num_nodes)
print("num edges:", data.edge_index.shape[1])
print("num features:", dataset.num_features)
print("num classes:", num_classes)
print("train nodes:", int(data.train_mask.sum()))
print("val nodes:", int(data.val_mask.sum()))
print("test nodes:", int(data.test_mask.sum()))

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])
num nodes: 19717
num edges: 88648
num features: 500
num classes: 3
train nodes: 60
val nodes: 500
test nodes: 1000


## a quick look at the graph itself
pubmed is way too big to draw all 19,717 nodes at once, so we grab a small connected sample and draw that instead - just to see what the graph actually looks like

function to grab a small connected sample of the graph, starting from one node and walking outward

In [7]:
def get_sample_subgraph(data_cpu, num_nodes=150, start_node=0):
    full_graph = to_networkx(data_cpu, to_undirected=True)
    sample_nodes = list(nx.bfs_tree(full_graph, start_node))[:num_nodes]
    return full_graph.subgraph(sample_nodes)

sample_graph = get_sample_subgraph(data_cpu, num_nodes=150, start_node=0)
print("sample has", sample_graph.number_of_nodes(), "nodes and", sample_graph.number_of_edges(), "edges")

sample has 150 nodes and 282 edges


## the two models

define a simple gcn model with two gcn layers

In [8]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x

define a simple gat model with two attention layers

In [9]:
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

write one function to train a model and one to test it, plus one that also returns the predictions themselves so we can compare them later

In [10]:
def train_model(model, edge_index, epochs=100):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    start = time.time()
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
    train_time = time.time() - start
    return train_time


@torch.no_grad()
def test_model(model, edge_index):
    model.eval()
    pred = model(data.x, edge_index).argmax(dim=1)
    acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
    return acc


@torch.no_grad()
def get_predictions(model, edge_index):
    model.eval()
    return model(data.x, edge_index).argmax(dim=1)

train gcn and gat on the clean graph (no noise yet) and keep their predictions for later

In [11]:
set_seed(0)
gcn_clean = GCN(dataset.num_features, 16, num_classes).to(device)
gcn_time = train_model(gcn_clean, data.edge_index)
gcn_clean_acc = test_model(gcn_clean, data.edge_index)
gcn_clean_pred = get_predictions(gcn_clean, data.edge_index)

set_seed(0)
gat_clean = GAT(dataset.num_features, 8, num_classes).to(device)
gat_time = train_model(gat_clean, data.edge_index)
gat_clean_acc = test_model(gat_clean, data.edge_index)
gat_clean_pred = get_predictions(gat_clean, data.edge_index)

print(f"gcn clean accuracy: {gcn_clean_acc:.4f}  |  training time: {gcn_time:.2f}s")
print(f"gat clean accuracy: {gat_clean_acc:.4f}  |  training time: {gat_time:.2f}s")

gcn clean accuracy: 0.7880  |  training time: 0.92s
gat clean accuracy: 0.7870  |  training time: 5.00s
